## NEU 171L Lab 1 - Analysis of MRI data
Jupyter is an interactive, web-based python interface. To run each cell use `ctrl+enter`. Three packages are used here: numpy (computation + sampling), pandas (data + statistics methods), and matplotlib (plotting).

### Learning Objectives
- Use descriptive statistics (mean, standard deviation, standard error of the mean) to quantify the basic features of data
- Use a **bootstrap** to build the sampling distribution of a mean, and report a 95% confidence interval from it
- Use a **permutation test** to ask whether a difference between two groups is larger than expected if the group labels were meaningless
- Understand what a **p-value** means with respect to the tested hypothesis


In [ ]:
# RUN THIS CELL to import python packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


### Step 1: Load in and clean the data

In [ ]:
# Step 1: Load in + clean the data
data = (pd.read_csv("oasis_cross-sectional.csv")
        .dropna(axis=0, subset=["CDR", "nWBV"]))   # drop participants without CDR or nWBV

dementia_data = data.loc[data.CDR > 0, :]          # nonzero clinical dementia rating
healthy_data  = data.loc[data.CDR == 0.0, :]       # general (healthy) population

print("Sample of dementia data:")
print(dementia_data.head(10), "\n")
print("Sample of healthy data:")
print(healthy_data.head(10), "\n")


We are only seeing the first 10 rows in each data set (that is what `.head(10)` does). 

### Step 2: Basic descriptive statistics

In this section you will summarize the normalized whole-brain volume (nWBV) of each group with three numbers: the **mean**, the **standard deviation (SD)**, and the **standard error of the mean (SEM)**.

The SEM measures how precisely we have estimated the *mean* — it shrinks as the sample gets larger:

$$\mathrm{SEM} = \frac{\mathrm{SD}}{\sqrt{N}}$$

where $N$ is the sample size. So before we compute the SEM, we first need $N$ for each group.

**First, the sample size.** Run the cell below to count participants in each group and save them as `n_healthy` and `n_dementia`. `.count()` counts the values in a column.

In [ ]:
# Sample size of each group (already complete -- just run it).
n_healthy   = healthy_data["nWBV"].count()
n_dementia  = dementia_data["nWBV"].count()

print("Sample size (N) of the healthy population:  %d" % n_healthy)
print("Sample size (N) of the dementia population: %d" % n_dementia)


**Next, the mean.** Calculate the **mean** nWBV for the healthy subjects. Apply the `mean` method to the "nWBV" column of `healthy_data` and print it. `.3f` prints three decimal places.

In [ ]:
# Mean of the healthy group (fill in the column name)
mean_healthy = healthy_data[...].mean()
print("Mean nWBV for the healthy population is %.3f" % (mean_healthy))


Calculate the **mean** nWBV for the dementia patients following the same pattern.

In [ ]:
# Mean of the dementia group
mean_dementia = ...
print(...)   # hint: use the same syntax as the healthy print line above, swapped to dementia


**Next, the standard deviation.** Calculate and print the **SD** of the nWBV for the healthy and dementia patients. The SD measures how spread out the individual values are. Method syntax is `.std()`.

In [ ]:
# Standard deviation of each group (fill in using the .std() method)
sd_healthy  = healthy_data["nWBV"] ...
sd_dementia = ...

print("SD of nWBV for the healthy population is  %.3f" % (sd_healthy))
print(...)   # hint: same syntax as the healthy SD print line above, swapped to dementia


**Finally, the standard error of the mean.** Combine the SD and sample size to get the SEM for the **healthy** group:

$$\mathrm{SEM} = \frac{\mathrm{SD}}{\sqrt{N}}$$

Syntax you need: `np.sqrt(x)` for the square root, and `/` for division. You already have `sd_healthy` and `n_healthy`.

In [ ]:
# Standard error of the mean for the HEALTHY group.
# SEM = SD / sqrt(N)
sem_healthy = ... / np.sqrt(...)

print("SEM of nWBV for the healthy population is %.4f" % (sem_healthy))


In [ ]:
# Plot the data as a histogram (just run this cell)
fig, ax = plt.subplots(1, figsize=(10,8))
ax.hist(healthy_data["nWBV"].values, rwidth=0.5, alpha=0.75, bins=10, label='healthy_data')
ax.hist(dementia_data["nWBV"].values, rwidth=0.5, alpha=0.75, bins=10, label='dementia_data')
ax.set_title("Distribution of nWBV for Dementia vs Healthy Population")
ax.set_xlabel("nWBV (A.U.)")
ax.set_ylabel("Count")
ax.legend()


### Step 2b: Bootstrapping the sampling distribution of the mean

The SEM you just calculated with the formula estimates how much the *mean* would bounce around if you repeatedly resampled the healthy population. A **bootstrap** lets us see that variability directly instead of trusting a formula, and lets us read a **95% confidence interval** straight off the resampled means.

The idea: treat the healthy sample as a stand-in for the whole population and draw many new resamples from it **with replacement** (the same participant can be picked more than once). Each resample gives a new mean. Thousands of these means form the **sampling distribution of the mean**.

The plot below marks four summaries of the mean's uncertainty, in two matching pairs:

- **SEM** (`SD / √N`) and **SD (bootstrap)** (the spread of the resampled means) are two ways to measure how much the mean varies — they should come out nearly equal.
- **95% CI calculated from SEM** (`mean ± 1.96 × SEM`) and **95% CI calculated from bootstrap** (the 2.5th–97.5th percentiles of the resampled means) are two ways to get the same interval — they should also nearly match.

Why bother with a bootstrap if we could just use SEM? The formula `mean ± 1.96 × SEM` only works for the mean. A median or a correlation has no such formula, but the bootstrap approach can give a CI for any such estimate in the same way. We use the mean here so we can check the two against each other as a demonstration.

**Your task:** each resample must be the **same size as the originally analyzed group**. Fill in the `n_to_draw_each_resample=` argument with the right value. The rest of the code is written for you. Hint: you already calculated sample sizes (n) in the first cell

Note: `replace=True` is what makes each resample draw **with replacement**.

In [ ]:
# Bootstrap the healthy group to build the sampling distribution of its mean.
n_iterations = 10000
bootstrap_means = np.zeros([n_iterations,])

for i in range(n_iterations):

    # How many values go into each resample? Fill in the correct variable
    n_to_draw_each_resample = ...

    # Draw a bootstrap resample from the healthy group, WITH replacement.
    current_resample = np.random.choice(a=healthy_data["nWBV"],
                                        size=n_to_draw_each_resample,
                                        replace=True)
    bootstrap_means[i] = current_resample.mean()

# The spread of the bootstrap means IS an estimate of the SEM:
print("SEM from the formula (SD / sqrt(N)):       %.4f" % sem_healthy)
sd_boot = bootstrap_means.std()                 # SD of the bootstrap means
print("Std of the bootstrap means (should match): %.4f" % sd_boot)

# This code calculates 95% confidence interval straight from the bootstrap distribution.
# because np.percentile(bootstrap_means, [2.5, 97.5]) returns the lower and upper edges.
ci_low, ci_high = np.percentile(bootstrap_means, [2.5, 97.5])
print("Bootstrap 95%% CI for the healthy mean nWBV: [%.4f, %.4f]" % (ci_low, ci_high))


ci_sem_low  = mean_healthy - 1.96 * sem_healthy     # 95% CI from the formula
ci_sem_high = mean_healthy + 1.96 * sem_healthy
print("SEM-based 95%% CI for the healthy mean nWBV: [%.4f, %.4f]" % (ci_sem_low, ci_sem_high))

### Plot the sampling distribution with four labeled lines drawn above the histogram:
SEM and SD (bootstrap)  -> two ways to measure the spread of the mean (should match)

the two 95% CIs         -> from the SEM formula vs from the bootstrap (should match)

(just run the cell below to make the figure)

In [ ]:
# (no edits needed, this part of the code will just run and produce a figure)
fig, ax = plt.subplots(1, figsize=(10, 8))
counts, _, _ = ax.hist(bootstrap_means, rwidth=0.9, alpha=0.5, bins=30,
                       label="Bootstrap distribution of the healthy mean")
ymax = counts.max()

# thin reference line at the mean (center of every interval below)
ax.axvline(mean_healthy, color="black", lw=1, alpha=0.4, label="_nolegend_")

# helper: draw a labeled horizontal interval [lo, hi] at height y
def interval(lo, hi, y, color, ls, label):
    ax.plot([lo, hi], [y, y], color=color, ls=ls, lw=2.5, marker="|",
            markersize=12, markeredgewidth=2.5, label=label)

# two measures of the spread of the mean (should be about equal):
interval(mean_healthy - sem_healthy, mean_healthy + sem_healthy, ymax*1.10, "C1", "-",  "SEM")
interval(mean_healthy - sd_boot,     mean_healthy + sd_boot,     ymax*1.20, "C1", "--", "SD (bootstrap means)")

# two 95% confidence intervals (should be about equal):
interval(ci_sem_low, ci_sem_high, ymax*1.35, "C3", "-",  "95% CI calculated from SEM")
interval(ci_low,     ci_high,     ymax*1.45, "C3", "--", "95% CI calculated from bootstrap")

ax.set_ylim(0, ymax*1.6)
ax.set_title("Bootstrap sampling distribution of the healthy mean nWBV")
ax.set_xlabel("Mean nWBV (A.U.)")
ax.set_ylabel("Count")
ax.legend(loc="upper right")

NameError: name 'plt' is not defined

### Step 3: Permutation test

We want to know whether healthy and dementia participants really differ in nWBV, or whether the difference could easily have arisen by chance.

**Null hypothesis:** the "healthy" and "dementia" labels are arbitrary — every participant's nWBV came from the *same* underlying distribution. If that were true, splitting participants into the two groups would be no different from shuffling the labels and dealing them into two random groups of the same sizes.

A **permutation test** builds the null distribution directly:
1. Compute the **observed difference** — the real difference in mean nWBV (healthy minus dementia).
2. Pool all participants, **shuffle the labels**, and re-split into random groups the size of the orignal groups.
3. Recompute the difference in means for that shuffled split — this gives you one value for the null distribution.
4. Repeat many times (here, 10,000) to build the whole null distribution.

**Why shuffle *without* replacement?** We are re-assigning labels to a fixed set of real participants; each appears exactly once per shuffle, just possibly in the other group. That differs from a **bootstrap**, which samples *with* replacement to imitate collecting brand-new data. `np.random.permutation` reshuffles the pooled values without replacement.

Fill in the sections marked `...`.

In [ ]:
# We shuffle the labels and record the difference in means each time (10,000 shuffles).
n_iterations = 10000

# Pool ALL participants' nWBV values (healthy + dementia) into one array.
pooled_nWBV = pd.concat([healthy_data.nWBV, dementia_data.nWBV]).values

# Size of the dementia group; after each shuffle the first n_dementia values are "dementia".
n_dementia = dementia_data.nWBV.count()

# OBSERVED difference: the real difference in group means (healthy minus dementia).
observed_diff = ...  # hint: "-" is the minus operator
print("Observed difference in mean nWBV (healthy - dementia): %.4f" % observed_diff)

perm_diffs = np.zeros([n_iterations,])
for i in range(n_iterations): # loop to repeat the shuffling 10,000 times

    # Shuffle all pooled values WITHOUT replacement.
    shuffled_nWBV = np.random.permutation(pooled_nWBV)

    # Deal the shuffled values into two new groups the same sizes as the real ones:
    # take the first n_dementia values as the new "dementia" group, and the rest as "healthy".
    perm_dementia = shuffled_nWBV[:n_dementia]   # [:n_dementia] means "the first n_dementia values"
    perm_healthy  = shuffled_nWBV[n_dementia:]   # [n_dementia:] means "everything after that"


    perm_healthy_mean = perm_healthy.mean()
    perm_dementia_mean = perm_dementia.mean()

    # Save the difference in means for this shuffle (healthy mean - dementia mean).
    perm_diffs[i] = ...   


In [ ]:
# Plot the NULL DISTRIBUTION: the differences in means produced by shuffling the labels.
# (just run this cell to produce the figure) 
fig, ax = plt.subplots(1, figsize=(10, 8))
ax.hist(perm_diffs, rwidth=0.9, alpha=0.5, bins=30,
        label="Null distribution of (healthy - dementia) mean differences\n(from shuffling the labels)")
ax.axvline(observed_diff, color="C1", lw=3,
           label="Observed difference (healthy - dementia)")
ax.set_title("Permutation test: null distribution of (healthy - dementia) mean nWBV")
ax.set_xlabel("Difference in mean nWBV, healthy - dementia (A.U.)")
ax.set_ylabel("Count")
ax.legend()


### Step 4: P-value

The histogram is the **null distribution**: all (healthy - dementia) differences expected **if the labels didn't matter**. The emphasized line is the observed difference we **actually measured**.

The **p-value**: *if the null were true, how often would chance alone produce a difference equal to or larger than the one observed?* It is the fraction of shuffled differences that are equal to or larger than the observed difference.

- A **small** p-value: the observed difference is far in the tail — chance rarely produces it.
- A **large** p-value: the observed difference sits in the thick of the null — chance produces it often.

**This is a one-sided test:** we expect dementia to be associated with *lower* nWBV, i.e. a *positive* (healthy - dementia) difference, so we count only shuffles whose difference is `>=` the observed one (larger in that one direction), not differences that are far off in either direction.

In [ ]:
# Use the ">=" operator to find which shuffled differences (perm_diffs) are equal to or larger than the 
# observed difference (observed_diff).
# This gives a True/False (Boolean) for each shuffle, which are then counted with .sum() on the line below.

boolean_more_extreme = ...   # hint: use the ">=" operator
count_more_extreme = boolean_more_extreme.sum() # counts the number of permuted differences that are equal or larger than the observed difference 

print("%d out of %d shuffled differences were equal to or larger than the observed difference."
      % (count_more_extreme, n_iterations))

# Divide that count_more_extreme by the total number of shuffles to get the p-value.
p_value = ...                # hint: "/" is the division operator, and "n_iterations" is the number of total shuffles
print("p-value (uncorrected): %0.4f" % p_value)


#### A small correction to the p-value

If **none** of your shuffles were equal to or larger than the observed difference, the formula above gives exactly **0** — claiming chance could *never* produce your result. But your real data is itself one arrangement of the labels that produced exactly that difference, so it is not impossible.

The fix: count the real data as one more arrangement. Add 1 to the numerator and 1 to the denominator:

$$p = \frac{\text{count} + 1}{n_{\text{iterations}} + 1}$$

This is the recommended permutation-test p-value. **Just run the cell below.**

In [ ]:
# Corrected permutation-test p-value: count the real data as one more arrangement.
# (just run this cell to calculate the corrected p-value)
p_value_corrected = (count_more_extreme + 1) / (n_iterations + 1)
print("p-value (uncorrected): %0.4f" % p_value)
print("p-value (corrected):   %0.4f" % p_value_corrected)


Answer the questions in the Lab 1 assignment to report and interpret the information in this notebook.